# 01 — Ingest

Fetch the raw source and land it in `data/raw/` **untouched**.

**Source:** Box Office Mojo — *Top Lifetime Adjusted Grosses* (domestic). The
`?adjust_gross_to=2022` query param makes the page return each film's
inflation-adjusted gross (in 2022 dollars) alongside its nominal gross,
estimated tickets sold, and release year. With that param the page is static
(no JS needed).

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
from io import StringIO
from src.ingest import load_config, fetch_html

cfg = load_config('config.yaml')
src = cfg['sources']['bom_adjusted']
url = src['url']
print('Source URL:', url)

## Fetch and cache the raw HTML
Save the page to `data/raw/` so the raw source is preserved and re-runs are
offline.

In [ ]:
raw_path = Path(cfg['paths']['data_raw']) / 'bom_top_lifetime_adjusted_2022.html'
raw_path.parent.mkdir(parents=True, exist_ok=True)

if raw_path.exists():
    print('Using cached raw HTML:', raw_path)
    html = raw_path.read_text(encoding='utf-8')
else:
    html = fetch_html(url)
    raw_path.write_text(html, encoding='utf-8')
    print('Saved raw HTML ->', raw_path)

print(f'{len(html):,} bytes')

## Parse the chart table
The page has a single table: Rank, Title, Adj. Lifetime Gross, Lifetime Gross,
Est. Num Tickets, Year.

In [ ]:
raw = pd.read_html(StringIO(html))[0]
print(raw.shape)
raw.head(10)

## Also fetch the WORLDWIDE chart
The adjusted chart above is **domestic only** (U.S. & Canada). To compare
domestic vs. international we also pull Box Office Mojo's worldwide chart,
which splits each film into worldwide / domestic / foreign gross.

In [ ]:
ww_url = cfg['sources']['bom_worldwide']['url']
ww_path = Path(cfg['paths']['data_raw']) / 'bom_ww_top_lifetime.html'
if ww_path.exists():
    ww_html = ww_path.read_text(encoding='utf-8')
else:
    ww_html = fetch_html(ww_url)
    ww_path.write_text(ww_html, encoding='utf-8')
ww_raw = pd.read_html(StringIO(ww_html))[0]
print(ww_raw.shape)
ww_raw.head(5)

## Also fetch the FOREIGN-LANGUAGE chart
The worldwide chart above only contains films big enough to rank *worldwide*,
so it systematically misses foreign films whose audience was mostly American
(a subtitled film that made most of its money in the U.S. has a modest
worldwide total and never appears there). To answer **“which foreign films
actually broke into the U.S. market?”** we pull Box Office Mojo's
**Foreign Language** genre chart directly — non-English-language films ranked
by U.S. & Canada (domestic) lifetime gross. NOMINAL (year-of-release) dollars.

In [ ]:
fl_url = cfg['sources']['bom_foreign']['url']
fl_path = Path(cfg['paths']['data_raw']) / 'bom_foreign_language.html'
if fl_path.exists():
    print('Using cached raw HTML:', fl_path)
    fl_html = fl_path.read_text(encoding='utf-8')
else:
    fl_html = fetch_html(fl_url)
    fl_path.write_text(fl_html, encoding='utf-8')
    print('Saved raw HTML ->', fl_path)
fl_raw = pd.read_html(StringIO(fl_html))[0]
print(fl_raw.shape)
fl_raw.head(10)

## Third source: film GENRE from the TMDB API

Box Office Mojo has **no genre column**, so genre is a *third data source* we
ingest here — from **TMDB** (The Movie Database) via its REST API.

**This step requires a free API key.** Get one at themoviedb.org (Settings →
API) and put it in the project's `.env` file (gitignored, never committed):

```
TMDB_API_KEY=your_key_here
```

For each film we call `GET /3/search/movie?query=<title>&year=<year>`, take the
first result, and map its `genre_ids` to names via `GET /3/genre/movie/list`.
The HTTP call itself lives in `src/ingest.py` → `tmdb_lookup_movie()` (per the
workspace norm that fetch logic lives in `src/`), and every response is **cached
as JSON in `data/raw/tmdb/`** so this raw data is preserved and re-runs are
offline. The next cell makes **one call explicitly** so the mechanics are visible.

In [ ]:
import requests
from src.ingest import load_env, tmdb_genre_map, tmdb_lookup_movie

load_env('.env')  # load TMDB_API_KEY from the gitignored .env
api_key = os.environ.get('TMDB_API_KEY')
assert api_key, 'TMDB_API_KEY missing from .env — see the note above'

# One raw TMDB call, spelled out, so you can see what the API returns:
demo = requests.get('https://api.themoviedb.org/3/search/movie',
                    params={'api_key': api_key, 'query': 'Avatar', 'year': 2009}, timeout=20)
top = demo.json()['results'][0]
print('HTTP', demo.status_code, '| result:', top['title'], top['release_date'])
print('raw genre_ids:', top['genre_ids'])
print('mapped genres:', [tmdb_genre_map(api_key)[i] for i in top['genre_ids']])

### Ingest genre for every film
Take the film list straight from the two raw box-office tables just fetched, and
look up each on TMDB (first run hits the API ~321 times, rate-limited; re-runs
read the JSON cache in `data/raw/tmdb/`). The raw genre records are saved to
`data/raw/tmdb_genres.parquet` — this is the ingested genre data that
`02-clean.ipynb` will load into DuckDB (no API calls happen in cleaning).

In [ ]:
# film list from the raw tables (title + year), deduped
adj_titles = raw[['Title', 'Year']].rename(columns={'Title':'title','Year':'release_year'})
ww_titles  = ww_raw[['Title', 'Year']].rename(columns={'Title':'title','Year':'release_year'})
titles = pd.concat([adj_titles, ww_titles]).drop_duplicates().reset_index(drop=True)
titles['release_year'] = titles['release_year'].astype(int)

recs = [tmdb_lookup_movie(r['title'], int(r['release_year']), api_key, cfg)
        for _, r in titles.iterrows()]
genre_raw = pd.DataFrame(recs)
genre_raw['genres_str'] = genre_raw['genres'].apply(lambda g: ', '.join(g) if g else None)
genre_raw.to_parquet(Path(cfg['paths']['data_raw']) / 'tmdb_genres.parquet')
print(f'{genre_raw["matched"].sum()}/{len(genre_raw)} films matched on TMDB')
genre_raw[['title','year','primary_genre','genres_str']].head(10)

### Also ingest each film's COUNTRY OF ORIGIN (TMDB)

Box Office Mojo's "domestic" means **U.S. & Canada** — which only equals a
film's *home market* for U.S.-made films. For a Chinese film like *Ne Zha 2*,
its money earned in China is bucketed as "foreign," which would make it look
like a film that conquered the world when it really conquered its home market.

So we also pull each film's **origin country** from TMDB's movie-detail
endpoint (`GET /3/movie/{id}`, via `tmdb_movie_country()` in `src/ingest.py`,
cached in `data/raw/tmdb/`). The `is_us` flag lets the domestic-vs-international
analysis stay consistent (home vs abroad) by restricting to U.S.-made films.
Non-U.S. films are kept in the data for a separate 'foreign films in the U.S.
market' angle.

In [ ]:
from src.ingest import tmdb_movie_country

country_rows = []
for r in genre_raw.to_dict('records'):
    if r.get('tmdb_id'):
        c = tmdb_movie_country(int(r['tmdb_id']), api_key, cfg)
        country_rows.append({'tmdb_id': int(r['tmdb_id']),
            'origin_country': ', '.join(c['origin_country']) if c['origin_country'] else None,
            'is_us': c['is_us']})
country_raw = pd.DataFrame(country_rows).drop_duplicates('tmdb_id')
country_raw.to_parquet(Path(cfg['paths']['data_raw']) / 'tmdb_countries.parquet')
print(f'{int(country_raw["is_us"].sum())} US-made / {len(country_raw)-int(country_raw["is_us"].sum())} non-US')
country_raw['origin_country'].value_counts().head(8)

### Origin country for the FOREIGN-LANGUAGE films (TMDB)

The foreign-language chart (`bom_foreign_language.html`) lists title + U.S.
gross but **no country**. For the “foreign films that broke into the U.S.”
chart we want to show *which country* each film came from, so we enrich each
title with its TMDB origin country — same two calls as above
(`tmdb_lookup_movie` → tmdb_id, then `tmdb_movie_country` → origin), both
cached in `data/raw/tmdb/`. Saved to `data/raw/tmdb_foreign_countries.parquet`.
We keep the **primary** (first-listed) origin country per film, since several
are co-productions (e.g. Crouching Tiger is TW/HK/CN/US).

In [ ]:
# Enrich the foreign-language titles with TMDB origin country.
fl_titles = fl_raw[['Title']].copy()
fl_titles['release_year'] = fl_raw['Release Date'].str[-4:].astype(int)
frows = []
for _, r in fl_titles.iterrows():
    m = tmdb_lookup_movie(r['Title'], int(r['release_year']), api_key, cfg)
    origin = None
    if m.get('tmdb_id'):
        c = tmdb_movie_country(int(m['tmdb_id']), api_key, cfg)
        oc = c.get('origin_country') or c.get('production_countries') or []
        origin = oc[0] if oc else None
    frows.append({'title': r['Title'], 'release_year': int(r['release_year']),
                  'tmdb_id': m.get('tmdb_id'), 'origin_country': origin})
foreign_countries = pd.DataFrame(frows)
foreign_countries.to_parquet(Path(cfg['paths']['data_raw']) / 'tmdb_foreign_countries.parquet')
print(f"{foreign_countries['origin_country'].notna().sum()}/{len(foreign_countries)} foreign titles got a country")
foreign_countries.head(15)

## Fourth source: the DOMESTIC all-time top 1000 (for origin analysis)

The Foreign Language chart above is a **language** grouping — it misses
English-language films made outside the U.S. (British, Australian, etc.). To
honestly answer **“which films *created outside the U.S.* did well in the
U.S.?”** we need the complete domestic universe, not a language slice.

So we pull Box Office Mojo's **all-time domestic (U.S. & Canada) lifetime
chart**, the deepest they publish (top ~1000), paginated by an `offset` param
in steps of 200. Each page is cached to `data/raw/`. Every title is then
enriched with its TMDB **origin country** (next cell), so the non-U.S. filter
is based on fact, not a curated guess. Rate-limited at 1.5s between page
fetches per workspace norms.

In [ ]:
import time
dom_src = cfg['sources']['bom_domestic_all']
dom_frames = []
for off in dom_src['offsets']:
    dpath = Path(cfg['paths']['data_raw']) / f'bom_domestic_all_{off:04d}.html'
    if dpath.exists():
        dhtml = dpath.read_text(encoding='utf-8')
    else:
        url = dom_src['url'] + (f'?offset={off}' if off else '')
        dhtml = fetch_html(url)
        dpath.write_text(dhtml, encoding='utf-8')
        print('fetched', url)
        time.sleep(dom_src.get('rate_limit_seconds', 1.5))
    dom_frames.append(pd.read_html(StringIO(dhtml))[0])
dom_all = pd.concat(dom_frames, ignore_index=True)
print('domestic all-time rows:', len(dom_all))
dom_all.head(5)

### Origin country for the full domestic top-1000 (TMDB)

Look up each domestic-chart title on TMDB (title+year → tmdb_id, then origin
country), reusing the same cached helpers. Many titles are already cached from
the genre pull above, so only new ones hit the API. We keep the **primary**
(first-listed) origin country — co-productions list multiple; the primary is a
documented convention, and the language-vs-origin comparison in `04-viz` makes
the co-production ambiguity visible rather than hiding it. Saved to
`data/raw/tmdb_domestic_countries.parquet`.

In [ ]:
dom_titles = dom_all[['Title', 'Year']].dropna().copy()
dom_titles['Year'] = dom_titles['Year'].astype(int)
drows = []
for _, r in dom_titles.iterrows():
    m = tmdb_lookup_movie(r['Title'], int(r['Year']), api_key, cfg)
    origin = None; is_us = None
    if m.get('tmdb_id'):
        c = tmdb_movie_country(int(m['tmdb_id']), api_key, cfg)
        oc = c.get('origin_country') or c.get('production_countries') or []
        origin = oc[0] if oc else None
        is_us = c.get('is_us')
    drows.append({'title': r['Title'], 'release_year': int(r['Year']),
                  'tmdb_id': m.get('tmdb_id'), 'origin_country': origin, 'is_us': is_us})
domestic_countries = pd.DataFrame(drows)
domestic_countries.to_parquet(Path(cfg['paths']['data_raw']) / 'tmdb_domestic_countries.parquet')
n_nonus = int((domestic_countries['is_us'] == False).sum())
print(f"{domestic_countries['origin_country'].notna().sum()}/{len(domestic_countries)} got a country; {n_nonus} non-US origin")
domestic_countries[domestic_countries['is_us'] == False].head(15)

### Production companies for the domestic top-1000 (TMDB) — the “Hollywood industry” proxy

Country-of-origin (above) marks Nolan's *The Odyssey* and the Harry Potter /
Bond films as U.K. — they follow production-company *registration* and filming
location, not which industry financed and released the film. A closer proxy
for **“made within the U.S./Hollywood industry”** is whether a **U.S.-registered
studio is among the production companies**. We pull `production_companies` from
the same `/movie/{id}` detail endpoint (`tmdb_movie_companies`, cached), and
flag `has_us_studio`. Saved to `data/raw/tmdb_domestic_companies.parquet`.

**This is a documented proxy, not a ground-truth field.** There is no public
“industry of origin” dataset; a U.S. major co-financing a culturally British
film is a genuine gray area. We record the definition so it's transparent, and
use it only for exploration (`04-viz` section F), not a headline claim.

In [ ]:
from src.ingest import tmdb_movie_companies

comp_rows = []
for r in domestic_countries.to_dict('records'):
    if r.get('tmdb_id') and pd.notna(r['tmdb_id']):
        cc = tmdb_movie_companies(int(r['tmdb_id']), api_key, cfg)
        comp_rows.append({'title': r['title'], 'release_year': r['release_year'],
            'tmdb_id': int(r['tmdb_id']),
            'companies_str': ', '.join(cc['companies'][:4]) if cc['companies'] else None,
            'company_countries': ', '.join(cc['company_countries']) if cc['company_countries'] else None,
            'has_us_studio': cc['has_us_studio']})
domestic_companies = pd.DataFrame(comp_rows)
domestic_companies.to_parquet(Path(cfg['paths']['data_raw']) / 'tmdb_domestic_companies.parquet')
n_nonhw = int((domestic_companies['has_us_studio'] == False).sum())
print(f'{len(domestic_companies)} films; {n_nonhw} with NO U.S. studio among producers')
domestic_companies[domestic_companies['has_us_studio'] == False].head(15)

## Fifth source: CPI-U (for consistent “today's dollars”)

Box Office Mojo's adjusted chart only offers a **2022** reference year and its
**ticket-price** method, and its foreign-language chart has no adjusted version
at all. To put every chart on ONE consistent inflation basis — **constant
“today's dollars”** — we pull the standard **CPI-U** and adjust each film's
nominal gross by its release year in `02-clean`. This replaces BOM's
ticket-price/2022 figures with a CPI method applied uniformly (documented in
SOURCES.md).

Source is **FRED series CPIAUCNS** (which mirrors the BLS CPI-U, U.S. city
average, all items, NSA, back to 1913) — downloaded as CSV, no API key. We use
FRED rather than the BLS public API because the BLS API v1 without a
registration key only returns ~3 recent years, and our films go back to 1937.
The fetch lives in `src/ingest.fetch_cpi_annual`, which caches the monthly CSV
and returns **annual averages** with a `complete` flag (the final year is
partial). We adjust to the latest **complete** year = “today's dollars.”

In [ ]:
from src.ingest import fetch_cpi_annual
cpi = fetch_cpi_annual(cfg)
base_year = int(cpi[cpi['complete']]['year'].max())  # latest COMPLETE annual CPI
print(f"CPI-U annual averages: {int(cpi['year'].min())}–{int(cpi['year'].max())} ({len(cpi)} years)")
print(f"Base year for 'today\u2019s dollars' = {base_year} (latest complete annual CPI); "
      f"{int(cpi['year'].max())} is partial and excluded as base.")
cpi.tail(6)

---
**Next:** `02-clean.ipynb` loads all three raw sources into DuckDB and cleans
them.

Nothing here modified data — the raw HTML and the TMDB JSON in `data/raw/` are
the untouched ingested sources.